In [ ]:
!git clone --branch Step-VIsualizer-+-Attention-Maps https://github.com/Salvatorevil03/ControlNet2026.git

Cloning into 'ControlNet2026'...
remote: Enumerating objects: 1444, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 1444 (delta 23), reused 22 (delta 22), pack-reused 1417 (from 2)
Receiving objects: 100% (1444/1444), 122.43 MiB | 23.42 MiB/s, done.
Resolving deltas: 100% (648/648), done.


In [ ]:
%cd /content/ControlNet2026

/content/ControlNet2026


In [ ]:
%%capture
!pip install -q gradio pytorch-lightning omegaconf einops test-tube streamlit streamlit-drawable-canvas webdataset kornia open_clip_torch invisible-watermark torchmetrics timm addict yapf basicsr imageio-ffmpeg opencv-contrib-python

In [ ]:
# @title ⬇️ Selezione Modelli ControlNet 1.5
# @markdown Spunta i modelli che desideri scaricare (attenzione: ogni modello pesa ~5.7 GB).

import os

# --- INIZIO FORM COLAB ---
Canny = True # @param {type:"boolean"}
Depth = True # @param {type:"boolean"}
HED = True # @param {type:"boolean"}
MLSD = True # @param {type:"boolean"}
Normal = True # @param {type:"boolean"}
OpenPose = True # @param {type:"boolean"}
Scribble = True # @param {type:"boolean"}
Segmentation = True # @param {type:"boolean"}
# --- FINE FORM COLAB ---

# Creiamo la lista dei modelli da scaricare in base alle tue spunte
models_to_download = []
if Canny: models_to_download.append("control_sd15_canny.pth")
if Depth: models_to_download.append("control_sd15_depth.pth")
if HED: models_to_download.append("control_sd15_hed.pth")
if MLSD: models_to_download.append("control_sd15_mlsd.pth")
if Normal: models_to_download.append("control_sd15_normal.pth")
if OpenPose: models_to_download.append("control_sd15_openpose.pth")
if Scribble: models_to_download.append("control_sd15_scribble.pth")
if Segmentation: models_to_download.append("control_sd15_seg.pth")

base_url = "https://huggingface.co/lllyasviel/ControlNet/resolve/main/models/"
model_dir = "/content/ControlNet2026/models"
 #/kaggle/working/ControlNet2026/models if running on Kaggle
# Assicuriamoci che la cartella esista
os.makedirs(model_dir, exist_ok=True)

if not models_to_download:
    print("⚠️ Nessun modello selezionato. Spunta almeno una casella.")
else:
    print(f"🚀 Inizio controllo e download di {len(models_to_download)} modelli...\n")

    for model in models_to_download:
        model_path = os.path.join(model_dir, model)
        if not os.path.exists(model_path):
            print(f"⬇️ Scaricando {model}...")
            # Scarica in modo silenzioso mostrando solo la progress bar
            !wget -q --show-progress -O "{model_path}" "{base_url}{model}"
        else:
            print(f"✅ {model} già presente, salto il download.")

    print("\n🎉 Download completato!")

🚀 Inizio controllo e download di 8 modelli...

✅ control_sd15_canny.pth già presente, salto il download.
⬇️ Scaricando control_sd15_depth.pth...
/content/ControlNet 100%[===================>]   5.32G  10.8MB/s    in 71s     
⬇️ Scaricando control_sd15_hed.pth...
/content/ControlNet 100%[===================>]   5.32G  93.0MB/s    in 1m 45s  
⬇️ Scaricando control_sd15_mlsd.pth...
/content/ControlNet 100%[===================>]   5.32G  27.1MB/s    in 98s     
⬇️ Scaricando control_sd15_normal.pth...
/content/ControlNet 100%[===================>]   5.32G   111MB/s    in 85s     
⬇️ Scaricando control_sd15_openpose.pth...
/content/ControlNet 100%[===================>]   5.32G  85.4MB/s    in 96s     
⬇️ Scaricando control_sd15_scribble.pth...
/content/ControlNet 100%[===================>]   5.32G  59.0MB/s    in 81s     
⬇️ Scaricando control_sd15_seg.pth...
/content/ControlNet 100%[===================>]   5.32G  38.8MB/s    in 87s     

🎉 Download completato!


In [ ]:
import os
import sys

"""
--- PATCH BASICSR vs TORCHVISION (V2) ---
Aggiorna l'import senza tentare di caricare la libreria rotta!
"""

print("Cerco il file difettoso di basicsr...")

# Troviamo la cartella dist-packages senza importare basicsr
dist_packages = [p for p in sys.path if 'dist-packages' in p][0]
degradations_file = os.path.join(dist_packages, 'basicsr', 'data', 'degradations.py')
print(degradations_file)

if os.path.exists(degradations_file):
    with open(degradations_file, 'r', encoding='utf-8') as f:
        content = f.read()

    vecchio_import = "from torchvision.transforms.functional_tensor import rgb_to_grayscale"
    nuovo_import = "from torchvision.transforms.functional import rgb_to_grayscale"

    if vecchio_import in content:
        content = content.replace(vecchio_import, nuovo_import)

        with open(degradations_file, 'w', encoding='utf-8') as f:
            f.write(content)
        print("✅ Patch applicata con successo! Libreria basicsr svecchiata.")
    else:
        print("⚠️ Patch già applicata o stringa non trovata.")
else:
    print(f"❌ Impossibile trovare il file: {degradations_file}")

Cerco il file difettoso di basicsr...
/usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py
✅ Patch applicata con successo! Libreria basicsr svecchiata.


In [ ]:
import os
import sys
import pytorch_lightning

# Fix per PyTorch Lightning 2.x
pl_path = os.path.dirname(pytorch_lightning.__file__)
dist_file = os.path.join(pl_path, 'utilities', 'distributed.py')

if not os.path.exists(dist_file):
    print("Applicazione patch di compatibilità per PyTorch Lightning...")
    os.makedirs(os.path.dirname(dist_file), exist_ok=True)
    with open(dist_file, 'w') as f:
        f.write('from pytorch_lightning.utilities.rank_zero import *')
    print("✅ Patch applicata.")

Applicazione patch di compatibilità per PyTorch Lightning...
✅ Patch applicata.


In [ ]:
%%capture
!pip install transformers

In [ ]:
!python gradio_canny2image.py

logging improved.
No module 'xformers'. Proceeding without it.
ControlLDM: Running in eps-prediction mode
DiffusionWrapper has 859.52 M params.
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 4, 32, 32) = 4096 dimensions.
making attention of type 'vanilla' with 512 in_channels
Loading weights: 100% 196/196 [00:00<00:00, 835.71it/s, Materializing param=text_model.final_layer_norm.weight]
Loaded model config from [./models/cldm_v15.yaml]
Loaded state_dict from [./models/control_sd15_canny.pth]
* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://da8923c50b53621570.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Seed set to 1836187117
Data shape for DDIM sampling is (1, 4, 64, 80), eta 0
Running DDIM Sampling with 20 timesteps
DDIM Sampler: 100% 20/20 [00:27<

In [ ]:
!python gradio_depth2image.py

logging improved.
/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name vit_base_resnet50_384 to current vit_base_r50_s16_384.orig_in21k_ft_in1k.
  model = create_fn(
No module 'xformers'. Proceeding without it.
ControlLDM: Running in eps-prediction mode
DiffusionWrapper has 859.52 M params.
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 4, 32, 32) = 4096 dimensions.
making attention of type 'vanilla' with 512 in_channels
Loading weights: 100% 196/196 [00:00<00:00, 844.91it/s, Materializing param=text_model.final_layer_norm.weight]
Loaded model config from [./models/cldm_v15.yaml]
Loaded state_dict from [./models/control_sd15_depth.pth]
* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://b1544b4a48a85feecc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to depl

In [ ]:
!python gradio_hed2image.py

logging improved.
Downloading: "https://huggingface.co/lllyasviel/Annotators/resolve/main/ControlNetHED.pth" to /content/ControlNet2026/annotator/ckpts/ControlNetHED.pth

100% 28.1M/28.1M [00:00<00:00, 46.4MB/s]
No module 'xformers'. Proceeding without it.
ControlLDM: Running in eps-prediction mode
DiffusionWrapper has 859.52 M params.
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 4, 32, 32) = 4096 dimensions.
making attention of type 'vanilla' with 512 in_channels
Loading weights: 100% 196/196 [00:00<00:00, 594.75it/s, Materializing param=text_model.final_layer_norm.weight]
Loaded model config from [./models/cldm_v15.yaml]
Loaded state_dict from [./models/control_sd15_hed.pth]
* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://0d36802769f03fd824.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging F

In [ ]:
!python gradio_normal2image.py

logging improved.
/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name vit_base_resnet50_384 to current vit_base_r50_s16_384.orig_in21k_ft_in1k.
  model = create_fn(
No module 'xformers'. Proceeding without it.
ControlLDM: Running in eps-prediction mode
DiffusionWrapper has 859.52 M params.
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 4, 32, 32) = 4096 dimensions.
making attention of type 'vanilla' with 512 in_channels
Loading weights: 100% 196/196 [00:00<00:00, 583.08it/s, Materializing param=text_model.final_layer_norm.weight]
Loaded model config from [./models/cldm_v15.yaml]
Loaded state_dict from [./models/control_sd15_normal.pth]
* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://37d2709e169ce121be.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to dep

In [ ]:
!python gradio_seg2image.py

logging improved.
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Downloading: "https://huggingface.co/lllyasviel/ControlNet/resolve/main/annotator/ckpts/upernet_global_small.pth" to /content/ControlNet2026/annotator/ckpts/upernet_global_small.pth

100% 197M/197M [00:02<00:00, 86.6MB/s]
Use Checkpoint: False
Checkpoint Number: [0, 0, 0, 0]
Use global window for all blocks in stage3
load checkpoint from local path: /content/ControlNet2026/annotator/ckpts/upernet_global_small.pth
No module 'xformers'. Proceeding without it.
ControlLDM: Running in eps-prediction mode
DiffusionWrapper has 859.52 M params.
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 4, 32, 32) = 4096 dimensions.
making attention of type 'vanilla' with 512 in_cha